# Week 9: Raster & Remote Sensing

This notebook covers:
- Working with satellite imagery using Rasterio
- Calculating NDVI (Normalized Difference Vegetation Index)
- Detecting vegetation change between two dates
- Zonal statistics

---

## Data options

| Option | Description |
|--------|-------------|
| **A. Sample data** | Use built-in synthetic satellite data. No download needed! |
| **B. Your own data** | Use real Sentinel-2 imagery you download from Copernicus |

**Recommendation:** Start with sample data to learn the workflow. Download real imagery later once you understand the concepts.

---

## Step 0: Set up environment

In [ ]:
# Detect environment and install packages
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing packages...")
    !pip install geopandas rasterio rasterstats -q
    print("Done!")
else:
    print("Running locally")
    print("Make sure you activated: conda activate intro-gis")

---

## Step 1: Set up folders

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW = Path("/content/drive/MyDrive/intro-gis/week09/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week09/data/processed")
else:
    RAW = Path("data/raw")
    PROCESSED = Path("data/processed")

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data folder: {RAW}")
print(f"Processed folder: {PROCESSED}")

---

## Step 2: Import libraries

In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box, Polygon
import rasterio
from rasterio.transform import from_bounds

print("Libraries imported!")

---

## Step 3: Load or generate data

This cell checks for local satellite imagery. If not found, it generates realistic sample data.

### Sample data (default)

The sample data simulates a **vegetation clearing event**:
- "Before" image: Forest with high NDVI
- "After" image: Same area with a cleared section (lower NDVI)

This lets you learn NDVI change detection without downloading large satellite files.

### Your own data

To use real Sentinel-2 imagery, place these files in `data/raw/`:
- `sentinel_before.tif` — Multi-band image (earlier date)
- `sentinel_after.tif` — Multi-band image (later date)  
- `aoi.geojson` — Your study area boundary
- `zones.geojson` — Zones for statistics (optional)

See the **Downloading Real Satellite Data** section at the end for detailed instructions.

In [ ]:
# Check if local files exist
local_before = RAW / "sentinel_before.tif"
local_after = RAW / "sentinel_after.tif"
local_aoi = RAW / "aoi.geojson"

if local_before.exists() and local_after.exists():
    print("Found local satellite imagery - using your own data")
    USE_SAMPLE_DATA = False
    
    # Load AOI if it exists
    if local_aoi.exists():
        aoi = gpd.read_file(local_aoi)
    else:
        print("Note: No aoi.geojson found, will use full raster extent")
        aoi = None
else:
    print("Local files not found - generating sample data...")
    print("(To use real imagery, add sentinel_before.tif and sentinel_after.tif to data/raw/)\n")
    USE_SAMPLE_DATA = True
    
    # Generate synthetic satellite data
    # This simulates a 10km x 10km area with realistic NDVI patterns
    
    np.random.seed(42)  # For reproducibility
    
    # Image dimensions (100m resolution, 10km x 10km)
    height, width = 100, 100
    
    # Bounding box (Sydney region for realism)
    minx, miny = 151.0, -33.9
    maxx, maxy = 151.1, -33.8
    
    # Create transform for georeferencing
    transform = from_bounds(minx, miny, maxx, maxy, width, height)
    
    # --- Generate "Before" image (healthy vegetation) ---
    # Base vegetation pattern with some natural variation
    base_ndvi = 0.6 + 0.15 * np.random.randn(height, width)
    
    # Add a river (low NDVI)
    river_y = np.sin(np.linspace(0, 2*np.pi, width)) * 10 + 50
    for x in range(width):
        y = int(river_y[x])
        if 0 <= y < height:
            base_ndvi[max(0,y-2):min(height,y+3), x] = -0.2
    
    # Add urban area (low NDVI) in corner
    base_ndvi[0:25, 0:25] = 0.1 + 0.05 * np.random.randn(25, 25)
    
    ndvi_before = np.clip(base_ndvi, -1, 1)
    
    # Convert NDVI to simulated Red and NIR bands
    # NDVI = (NIR - Red) / (NIR + Red)
    # Solving: NIR = Red * (1 + NDVI) / (1 - NDVI)
    red_before = 1000 + 500 * np.random.randn(height, width)
    red_before = np.clip(red_before, 100, 3000).astype(np.uint16)
    nir_before = (red_before * (1 + ndvi_before) / (1 - ndvi_before + 0.001)).astype(np.uint16)
    nir_before = np.clip(nir_before, 100, 10000).astype(np.uint16)
    
    # --- Generate "After" image (with clearing) ---
    ndvi_after = ndvi_before.copy()
    
    # Simulate vegetation clearing (rectangular area)
    clearing_y1, clearing_y2 = 40, 70
    clearing_x1, clearing_x2 = 50, 80
    ndvi_after[clearing_y1:clearing_y2, clearing_x1:clearing_x2] = 0.15 + 0.05 * np.random.randn(
        clearing_y2 - clearing_y1, clearing_x2 - clearing_x1
    )
    ndvi_after = np.clip(ndvi_after, -1, 1)
    
    red_after = 1000 + 500 * np.random.randn(height, width)
    red_after = np.clip(red_after, 100, 3000).astype(np.uint16)
    nir_after = (red_after * (1 + ndvi_after) / (1 - ndvi_after + 0.001)).astype(np.uint16)
    nir_after = np.clip(nir_after, 100, 10000).astype(np.uint16)
    
    # Store for later use
    sample_data = {
        'red_before': red_before,
        'nir_before': nir_before,
        'red_after': red_after,
        'nir_after': nir_after,
        'transform': transform,
        'crs': 'EPSG:4326',
        'bounds': (minx, miny, maxx, maxy)
    }
    
    # Create AOI and zones
    aoi = gpd.GeoDataFrame(
        {'name': ['Study Area']},
        geometry=[box(minx, miny, maxx, maxy)],
        crs='EPSG:4326'
    )
    
    # Create analysis zones (4 quadrants)
    mid_x = (minx + maxx) / 2
    mid_y = (miny + maxy) / 2
    zones = gpd.GeoDataFrame({
        'name': ['Northwest', 'Northeast', 'Southwest', 'Southeast'],
        'geometry': [
            box(minx, mid_y, mid_x, maxy),
            box(mid_x, mid_y, maxx, maxy),
            box(minx, miny, mid_x, mid_y),
            box(mid_x, miny, maxx, mid_y)
        ]
    }, crs='EPSG:4326')
    
    print("Generated sample data:")
    print(f"  Image size: {width} x {height} pixels")
    print(f"  Simulated area: Sydney region ({minx}, {miny}) to ({maxx}, {maxy})")
    print(f"  Scenario: Vegetation clearing in southeast quadrant")

---

## Step 4: Understanding NDVI

### What is NDVI?

**NDVI (Normalized Difference Vegetation Index)** measures vegetation health using satellite imagery. It exploits a key property of plants: **healthy vegetation absorbs red light for photosynthesis but strongly reflects near-infrared (NIR) light**.

### The formula

```
NDVI = (NIR - Red) / (NIR + Red)
```

### Interpreting NDVI values

| NDVI Range | What it means |
|------------|---------------|
| 0.6 to 1.0 | Dense, healthy vegetation (forests, crops at peak) |
| 0.3 to 0.6 | Moderate vegetation (shrubs, grassland) |
| 0.1 to 0.3 | Sparse vegetation or stressed plants |
| -0.1 to 0.1 | Bare soil, rock, urban areas |
| -1.0 to -0.1 | Water, snow, clouds |

### Which satellite bands?

| Satellite | Red Band | NIR Band |
|-----------|----------|----------|
| **Sentinel-2** | Band 4 (index 3) | Band 8 (index 7) |
| **Landsat 8/9** | Band 4 (index 3) | Band 5 (index 4) |

---

## Step 5: Calculate NDVI

In [ ]:
def calculate_ndvi(red, nir):
    """
    Calculate NDVI from red and NIR bands.
    
    Parameters:
    - red: 2D array of red band values
    - nir: 2D array of NIR band values
    
    Returns:
    - ndvi: 2D array with values from -1 to 1
    """
    red = red.astype(float)
    nir = nir.astype(float)
    
    # Avoid division by zero
    np.seterr(divide='ignore', invalid='ignore')
    ndvi = (nir - red) / (nir + red)
    ndvi[~np.isfinite(ndvi)] = np.nan
    
    return ndvi

if USE_SAMPLE_DATA:
    # Use generated sample data
    ndvi_before = calculate_ndvi(sample_data['red_before'], sample_data['nir_before'])
    ndvi_after = calculate_ndvi(sample_data['red_after'], sample_data['nir_after'])
    raster_transform = sample_data['transform']
else:
    # Load from real files
    # Adjust band indices for your satellite!
    RED_BAND = 3  # Sentinel-2 Band 4
    NIR_BAND = 7  # Sentinel-2 Band 8
    
    with rasterio.open(local_before) as src:
        red_before = src.read(RED_BAND + 1)  # rasterio uses 1-based indexing
        nir_before = src.read(NIR_BAND + 1)
        raster_transform = src.transform
    
    with rasterio.open(local_after) as src:
        red_after = src.read(RED_BAND + 1)
        nir_after = src.read(NIR_BAND + 1)
    
    ndvi_before = calculate_ndvi(red_before, nir_before)
    ndvi_after = calculate_ndvi(red_after, nir_after)

print("NDVI Before:")
print(f"  Min:  {np.nanmin(ndvi_before):.3f}")
print(f"  Max:  {np.nanmax(ndvi_before):.3f}")
print(f"  Mean: {np.nanmean(ndvi_before):.3f}")

print("\nNDVI After:")
print(f"  Min:  {np.nanmin(ndvi_after):.3f}")
print(f"  Max:  {np.nanmax(ndvi_after):.3f}")
print(f"  Mean: {np.nanmean(ndvi_after):.3f}")

---

## Step 6: Visualize NDVI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# NDVI Before
im1 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title("NDVI - Before")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], shrink=0.8, label="NDVI")

# NDVI After
im2 = axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI - After")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], shrink=0.8, label="NDVI")

plt.suptitle("NDVI: Green = healthy vegetation, Red/Yellow = bare/stressed", y=1.02)
plt.tight_layout()
plt.show()

---

## Step 7: Calculate NDVI change

Detect vegetation change by comparing NDVI between dates.

- **Positive change** = vegetation increased (greening, regrowth)
- **Negative change** = vegetation decreased (clearing, drought, fire)

In [ ]:
# Calculate NDVI change (after minus before)
ndvi_change = ndvi_after - ndvi_before

print("NDVI Change statistics:")
print(f"  Mean:  {np.nanmean(ndvi_change):.3f}")
print(f"  Min:   {np.nanmin(ndvi_change):.3f}")
print(f"  Max:   {np.nanmax(ndvi_change):.3f}")

# Count pixels with significant change
significant_decrease = np.sum(ndvi_change < -0.1)
significant_increase = np.sum(ndvi_change > 0.1)
total_pixels = np.sum(~np.isnan(ndvi_change))

print(f"\nPixels with significant vegetation loss (< -0.1):  {significant_decrease:,} ({100*significant_decrease/total_pixels:.1f}%)")
print(f"Pixels with significant vegetation gain (> +0.1):  {significant_increase:,} ({100*significant_increase/total_pixels:.1f}%)")

---

## Step 8: Visualize change

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# NDVI Before
im0 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title("NDVI Before")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# NDVI After
im1 = axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI After")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# NDVI Change
im2 = axes[2].imshow(ndvi_change, cmap="RdYlGn", vmin=-0.5, vmax=0.5)
axes[2].set_title("NDVI Change\n(Green = gain, Red = loss)")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], shrink=0.8, label="Change")

plt.tight_layout()
plt.show()

if USE_SAMPLE_DATA:
    print("\nNote: The red area in the 'After' image shows simulated vegetation clearing.")

---

## Step 9: Zonal statistics

Summarize NDVI change by zone (like QGIS Zonal Statistics tool).

In [ ]:
from rasterstats import zonal_stats

# Load zones if using own data
if not USE_SAMPLE_DATA:
    zones_path = RAW / "zones.geojson"
    if zones_path.exists():
        zones = gpd.read_file(zones_path)
    else:
        print("No zones.geojson found - creating simple grid zones")
        # Create zones from AOI
        bounds = aoi.total_bounds if aoi is not None else (0, 0, 1, 1)
        minx, miny, maxx, maxy = bounds
        mid_x, mid_y = (minx + maxx) / 2, (miny + maxy) / 2
        zones = gpd.GeoDataFrame({
            'name': ['NW', 'NE', 'SW', 'SE'],
            'geometry': [
                box(minx, mid_y, mid_x, maxy),
                box(mid_x, mid_y, maxx, maxy),
                box(minx, miny, mid_x, mid_y),
                box(mid_x, miny, maxx, mid_y)
            ]
        }, crs='EPSG:4326')

# Calculate zonal statistics
stats = zonal_stats(
    zones,
    ndvi_change,
    affine=raster_transform,
    stats=["mean", "min", "max", "count"],
    nodata=np.nan
)

# Add to GeoDataFrame
zones["ndvi_change_mean"] = [s["mean"] for s in stats]
zones["ndvi_change_min"] = [s["min"] for s in stats]
zones["ndvi_change_max"] = [s["max"] for s in stats]
zones["pixel_count"] = [s["count"] for s in stats]

print("Zonal Statistics:")
zones[["name", "ndvi_change_mean", "ndvi_change_min", "ndvi_change_max", "pixel_count"]]

---

## Step 10: Map zonal results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

zones.plot(
    column="ndvi_change_mean",
    cmap="RdYlGn",
    legend=True,
    legend_kwds={"label": "Mean NDVI Change"},
    edgecolor="black",
    linewidth=2,
    ax=ax
)

# Add zone labels
for idx, row in zones.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        f"{row['name']}\n{row['ndvi_change_mean']:.3f}",
        xy=(centroid.x, centroid.y),
        ha='center', va='center',
        fontsize=10, fontweight='bold'
    )

ax.set_title("Mean NDVI Change by Zone\n(Green = vegetation gain, Red = vegetation loss)")
ax.set_axis_off()
plt.show()

---

## Step 11: Export results

In [ ]:
# Save zones with statistics
zones.to_file(PROCESSED / "zones_ndvi_change.gpkg", driver="GPKG")
print(f"Saved zones to: {PROCESSED / 'zones_ndvi_change.gpkg'}")

# Save NDVI change as GeoTIFF
if USE_SAMPLE_DATA:
    output_tif = PROCESSED / "ndvi_change.tif"
    with rasterio.open(
        output_tif,
        'w',
        driver='GTiff',
        height=ndvi_change.shape[0],
        width=ndvi_change.shape[1],
        count=1,
        dtype=ndvi_change.dtype,
        crs=sample_data['crs'],
        transform=sample_data['transform']
    ) as dst:
        dst.write(ndvi_change, 1)
    print(f"Saved NDVI change raster to: {output_tif}")

print("\nYou can now open these files in QGIS!")

---

## Done!

You've completed vegetation change detection:

1. Loaded/generated satellite imagery
2. Calculated NDVI for both dates
3. Detected vegetation change
4. Summarized change by zones
5. Exported results for QGIS

---

## Downloading Real Satellite Data

Ready to analyze real imagery? Here's how to download Sentinel-2 data from Copernicus.

### Step 1: Create a free account

1. Go to [Copernicus Browser](https://browser.dataspace.copernicus.eu/)
2. Click **Login** → **Register**
3. Fill in your details and verify your email

### Step 2: Find your study area

1. Use the map to navigate to your area of interest
2. Click the **Draw** tool (polygon icon) in the left panel
3. Draw a rectangle or polygon around your study area
4. **Keep it small** — a 10km × 10km area is plenty for learning

### Step 3: Filter images

In the left panel:

1. **Data source:** Select **Sentinel-2** → **Sentinel-2 L2A** (atmospherically corrected)
2. **Time range:** Set your date range
   - For change detection, you need TWO dates (e.g., January and June)
   - Choose dates 3-6 months apart to see vegetation change
3. **Cloud cover:** Set maximum to **10%** (fewer clouds = clearer image)
4. Click **Search**

### Step 4: Preview and select

1. Browse the results — click each to preview
2. Look for images with:
   - Low cloud cover over your specific area
   - Good visibility (not hazy)
3. Select ONE image for your "before" date

### Step 5: Download

1. Click on your selected image
2. Click the **Download** icon
3. Choose **Full product** (this downloads all bands)
4. Repeat for your "after" date image

### Step 6: Extract and prepare

Downloaded files are large ZIPs (~800MB each). Inside you'll find:

```
S2A_MSIL2A_.../
└── GRANULE/
    └── L2A_.../
        └── IMG_DATA/
            └── R10m/           ← 10m resolution bands
                ├── *_B02_10m.jp2   (Blue)
                ├── *_B03_10m.jp2   (Green)
                ├── *_B04_10m.jp2   (Red)     ← You need this
                └── *_B08_10m.jp2   (NIR)     ← And this
```

For this notebook, you need the **Red (B04)** and **NIR (B08)** bands.

### Step 7: Create multi-band GeoTIFF (optional)

To combine bands into a single file, use QGIS:

1. **Raster → Miscellaneous → Merge**
2. Add B04 and B08 as input layers
3. Check "Place each input file into a separate band"
4. Save as `sentinel_before.tif`
5. Repeat for the "after" date

---

### Alternative: Use pre-processed imagery

If Sentinel-2 downloads are too complex, try:

- **Google Earth Engine** — cloud-based processing, no downloads needed
- **Planetary Computer** — Microsoft's free satellite data platform
- **OpenAerialMap** — drone imagery for smaller areas

---

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`